# 14b Gu et al.'s weighted ensemble on NSD

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. Gu's ensemble (per region as in their code, per region pooled, per vertex) against the map, and the map fitted on
top of each, on P-7's people and budgets.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '14b-guensemble', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "ks": [250, 500], "draws": [13, 14, 15], "wall_hours": 11.0}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.baselines import gu_by_group
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in ("alexnet",):
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def gu_vertex(src_k, y_k, src_t):
    # per vertex OLS of the person's response on the donors' predictions (with an intercept), batched over vertices
    xm, ym = src_k.mean(1, keepdims=True), y_k.mean(0)
    xc = src_k - xm                                   # (donors, k, V)
    G = np.einsum("ikv,jkv->vij", xc, xc)             # (V, donors, donors)
    G += 1e-6 * np.trace(G, axis1=1, axis2=2)[:, None, None] * np.eye(len(src_k))[None]
    b = np.einsum("ikv,kv->vi", xc, y_k - ym)         # (V, donors)
    w = np.linalg.solve(G, b[..., None])[..., 0]      # (V, donors)
    return np.einsum("vi,imv->mv", w, src_t - xm) + ym

rows = []
stopped = None
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    x_pool, x_test = feats("alexnet", ids[s][pool_rows[s]]), feats("alexnet", test_ids)
    # donors' predictions on the test images kept (7 x 872 x V float32, about 1 GB); on the pool only their running
    # mean (all seven on about 9,000 images would be about 10 GB), the labelled rows recomputed per draw
    src_test = np.stack([ols_pred(BETA[("alexnet", d)], x_test).astype(np.float32) for d in donors])  # (7, 872, V)
    avg_pool = np.zeros((len(x_pool), V), np.float32)
    for d in donors:
        avg_pool += ols_pred(BETA[("alexnet", d)], x_pool).astype(np.float32)
    avg_pool /= len(donors)
    avg_test = src_test.mean(0).astype(np.float64)
    mean_b, B = lm.prediction_basis(avg_pool, FIXED["K"])
    nl = len(common["lh"])
    group = np.where(np.arange(V) < nl, -1, -2)
    for ci, cls in enumerate(FIXED["roi_classes"]):  # first label wins, in the classes' order
        lab = rois[(s, cls)]
        free = (group < 0) & (lab > 0)
        group[free] = 1000 * (ci + 1) + lab[free]
    def add(method, k, d, pred):
        rows.append({"subject": s, "method": method, "k": k, "draw": d, "enc_r": nc.encoding_r(pred, meas),
                     "top1": float(np.mean([nc.identify_top1(pred[g], meas[g]) for g in GALS]))})
    add("avg", 0, "-", avg_test)
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks"]:
            idx = order[:k]
            y_k, a_k = y_pool[idx].astype(np.float64), avg_pool[idx].astype(np.float64)
            src_k = np.stack([ols_pred(BETA[("alexnet", dn)], x_pool[idx]) for dn in donors])  # (7, k, V) float64
            model = lm.fit_lowrank_map(a_k, y_k, mean_b, B, seed=d)
            add("map", k, d, lm.apply_lowrank_map(model, avg_test, mean_b, B))
            gu = {"gu_region_mean": gu_by_group(src_k, y_k, src_test, group, "mean"),
                  "gu_region_pooled": gu_by_group(src_k, y_k, src_test, group, "pooled"),
                  "gu_vertex": gu_vertex(src_k, y_k, src_test)}
            gu_k = {"gu_region_mean": gu_by_group(src_k, y_k, src_k, group, "mean"),
                    "gu_region_pooled": gu_by_group(src_k, y_k, src_k, group, "pooled"),
                    "gu_vertex": gu_vertex(src_k, y_k, src_k)}
            for name, pr in gu.items():
                add(name, k, d, np.asarray(pr, np.float64))
                # the map on top of Gu's ensemble: the same low-rank ridge, on the residual of Gu's prediction
                scores_k = (a_k - mean_b) @ B
                fit = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(scores_k, y_k - np.asarray(gu_k[name], np.float64))
                add(f"map_on_{name}", k, d, np.asarray(pr, np.float64) + fit.predict((avg_test - mean_b) @ B))
        if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
            stopped = f"wall clock after {s} draw {d}"
            break
    del src_test, avg_pool
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "14b_scores.csv", index=False)
    print(s, {r["method"]: round(r["enc_r"], 4) for r in rows if r["subject"] == s and r["k"] == 500 and r["draw"] == 13},
          f"| {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
    if stopped:
        print("STOPPED:", stopped)
        break

In [ ]:
frame = pd.DataFrame(rows)
done = [s for s in SUBJECTS if (frame.subject == s).any()]
def per_subject(method, metric):
    return frame[frame.method == method].groupby("subject")[metric].mean().reindex(done).to_numpy()
def desc(x):
    return {"mean": float(np.mean(x)), "positive": int((x > 0).sum()), "n": int(len(x))}
res = {"post_hoc": "development: Gu et al.'s ensemble on NSD", "levels": {}, "contrasts": {}}
for m in sorted(set(frame.method)):
    res["levels"][m] = {"enc_r": float(np.mean(per_subject(m, "enc_r"))), "top1": float(np.mean(per_subject(m, "top1")))}
for g in ("gu_region_mean", "gu_region_pooled", "gu_vertex"):
    for a, b in (("map", g), (f"map_on_{g}", g), (g, "avg"), (f"map_on_{g}", "map")):
        for metric in ("enc_r", "top1"):
            res["contrasts"][f"{a} - {b}|{metric}"] = desc(per_subject(a, metric) - per_subject(b, metric))
res["check_10c_C1"] = float(np.mean(per_subject("map", "enc_r") - per_subject("avg", "enc_r")))
res["stopped"] = stopped
(C.METRICS_DIR / "14b_summary.json").write_text(json.dumps(res, indent=2))
print("14B_SUMMARY_JSON " + json.dumps(res))